# TrialScope — Data Validation

**Project:** TrialScope  
**Phase:** 2 — Data Validation  
**Data source:** ClinicalTrials.gov API v2

## Objective

This notebook validates the observations made during the API exploration phase before implementing the final ingestion pipeline.

It is **not** the production extractor. Its purpose is to verify:

- module and field presence;
- missing and optional fields;
- categorical values;
- partial-date precision;
- age values and units;
- relationship cardinalities;
- locations and outcomes;
- relevant source fields intentionally outside the current relational model.

The results should be used to validate the Data Model Notes and PostgreSQL schema.


## 1. Imports


In [1]:
import requests
from collections import Counter
import re

API_URL = "https://clinicaltrials.gov/api/v2/studies"
TARGET_STUDIES = 200
PAGE_SIZE = 100
TIMEOUT = 30


## 2. Retrieve a validation sample

The notebook retrieves a fixed sample and handles API pagination. This is for validation, not the final load.


In [2]:
def get_studies(limit=200, page_size=100):
    studies = []
    page_token = None

    while len(studies) < limit:
        params = {"pageSize": min(page_size, limit - len(studies))}

        if page_token:
            params["pageToken"] = page_token

        response = requests.get(API_URL, params=params, timeout=TIMEOUT)
        response.raise_for_status()

        data = response.json()
        studies.extend(data.get("studies", []))

        page_token = data.get("nextPageToken")
        if not page_token:
            break

    return studies[:limit]


studies = get_studies(TARGET_STUDIES)
print("Studies retrieved:", len(studies))


Studies retrieved: 200


## 3. Basic structure


In [3]:
def protocol_of(study):
    return study.get("protocolSection", {})

protocols = [protocol_of(study) for study in studies]

print("Records with protocolSection:",
      sum(bool(protocol) for protocol in protocols),
      "/", len(protocols))

module_counts = Counter()
for protocol in protocols:
    module_counts.update(protocol.keys())

print("\nModules observed:")
for module, count in sorted(module_counts.items()):
    print(f"{module:35} {count:3}/{len(protocols)}")


Records with protocolSection: 200 / 200

Modules observed:
armsInterventionsModule             191/200
conditionsModule                    200/200
contactsLocationsModule             191/200
descriptionModule                   200/200
designModule                        200/200
eligibilityModule                   200/200
identificationModule                200/200
ipdSharingStatementModule            96/200
outcomesModule                      192/200
oversightModule                     191/200
referencesModule                     70/200
sponsorCollaboratorsModule          200/200
statusModule                        200/200


## 4. Relevant module presence


In [4]:
RELEVANT_MODULES = [
    "identificationModule",
    "statusModule",
    "designModule",
    "conditionsModule",
    "sponsorCollaboratorsModule",
    "armsInterventionsModule",
    "eligibilityModule",
    "contactsLocationsModule",
    "outcomesModule",
]

for module in RELEVANT_MODULES:
    present = sum(module in protocol for protocol in protocols)
    print(f"{module:35} present={present:3} | missing={len(protocols)-present:3}")


identificationModule                present=200 | missing=  0
statusModule                        present=200 | missing=  0
designModule                        present=200 | missing=  0
conditionsModule                    present=200 | missing=  0
sponsorCollaboratorsModule          present=200 | missing=  0
armsInterventionsModule             present=191 | missing=  9
eligibilityModule                   present=200 | missing=  0
contactsLocationsModule             present=191 | missing=  9
outcomesModule                      present=192 | missing=  8


## 5. Presence of fields used by the current model


In [5]:
FIELD_PATHS = {
    "nctId": ("identificationModule", "nctId"),
    "briefTitle": ("identificationModule", "briefTitle"),
    "officialTitle": ("identificationModule", "officialTitle"),
    "overallStatus": ("statusModule", "overallStatus"),
    "startDateStruct": ("statusModule", "startDateStruct"),
    "primaryCompletionDateStruct": ("statusModule", "primaryCompletionDateStruct"),
    "completionDateStruct": ("statusModule", "completionDateStruct"),
    "studyType": ("designModule", "studyType"),
    "phases": ("designModule", "phases"),
    "enrollmentInfo": ("designModule", "enrollmentInfo"),
    "conditions": ("conditionsModule", "conditions"),
    "leadSponsor": ("sponsorCollaboratorsModule", "leadSponsor"),
    "collaborators": ("sponsorCollaboratorsModule", "collaborators"),
    "interventions": ("armsInterventionsModule", "interventions"),
    "armGroups": ("armsInterventionsModule", "armGroups"),
    "minimumAge": ("eligibilityModule", "minimumAge"),
    "maximumAge": ("eligibilityModule", "maximumAge"),
    "sex": ("eligibilityModule", "sex"),
    "healthyVolunteers": ("eligibilityModule", "healthyVolunteers"),
    "stdAges": ("eligibilityModule", "stdAges"),
    "eligibilityCriteria": ("eligibilityModule", "eligibilityCriteria"),
    "locations": ("contactsLocationsModule", "locations"),
    "primaryOutcomes": ("outcomesModule", "primaryOutcomes"),
    "secondaryOutcomes": ("outcomesModule", "secondaryOutcomes"),
}

def get_nested(obj, path):
    value = obj
    for key in path:
        if not isinstance(value, dict):
            return None
        value = value.get(key)
    return value

for field, path in FIELD_PATHS.items():
    present = sum(get_nested(protocol, path) is not None for protocol in protocols)
    print(f"{field:35} present={present:3} | missing={len(protocols)-present:3}")


nctId                               present=200 | missing=  0
briefTitle                          present=200 | missing=  0
officialTitle                       present=199 | missing=  1
overallStatus                       present=200 | missing=  0
startDateStruct                     present=199 | missing=  1
primaryCompletionDateStruct         present=193 | missing=  7
completionDateStruct                present=199 | missing=  1
studyType                           present=200 | missing=  0
phases                              present=156 | missing= 44
enrollmentInfo                      present=200 | missing=  0
conditions                          present=200 | missing=  0
leadSponsor                         present=200 | missing=  0
collaborators                       present= 56 | missing=144
interventions                       present=182 | missing= 18
armGroups                           present=179 | missing= 21
minimumAge                          present=189 | missing= 11
maximumA

## 6. Categorical values


In [6]:
def collect_values(path, list_field=False):
    values = []
    for protocol in protocols:
        value = get_nested(protocol, path)
        if value is None:
            continue
        if list_field:
            values.extend(value)
        else:
            values.append(value)
    return Counter(values)

categorical_fields = {
    "studyType": (("designModule", "studyType"), False),
    "phases": (("designModule", "phases"), True),
    "overallStatus": (("statusModule", "overallStatus"), False),
    "sex": (("eligibilityModule", "sex"), False),
}

for name, (path, is_list) in categorical_fields.items():
    print(f"\n{name}")
    for value, count in collect_values(path, is_list).most_common():
        print(f"  {value}: {count}")



studyType
  INTERVENTIONAL: 156
  OBSERVATIONAL: 44

phases
  NA: 82
  PHASE2: 31
  PHASE1: 23
  PHASE3: 18
  PHASE4: 7
  EARLY_PHASE1: 3

overallStatus
  COMPLETED: 104
  UNKNOWN: 45
  RECRUITING: 21
  NOT_YET_RECRUITING: 9
  TERMINATED: 9
  ACTIVE_NOT_RECRUITING: 6
  WITHDRAWN: 4
  SUSPENDED: 1
  ENROLLING_BY_INVITATION: 1

sex
  ALL: 178
  FEMALE: 14
  MALE: 8


## 7. Phase validation

`NA` is not treated as a real clinical phase. The validation also checks whether `NA` ever appears together with a real phase.


In [7]:
phase_lists = []

for protocol in protocols:
    phases = get_nested(protocol, ("designModule", "phases"))
    if phases is not None:
        phase_lists.append(phases)

na_only = sum(phases == ["NA"] for phases in phase_lists)
na_with_real_phase = sum(
    "NA" in phases and any(phase != "NA" for phase in phases)
    for phases in phase_lists
)

print("Trials with phases field:", len(phase_lists))
print("NA only:", na_only)
print("NA combined with real phase:", na_with_real_phase)

print("\nReal phase values:")
for phase, count in Counter(
    phase for phases in phase_lists for phase in phases if phase != "NA"
).most_common():
    print(f"  {phase}: {count}")


Trials with phases field: 156
NA only: 82
NA combined with real phase: 0

Real phase values:
  PHASE2: 31
  PHASE1: 23
  PHASE3: 18
  PHASE4: 7
  EARLY_PHASE1: 3


## 8. Date validation

The model preserves the source date precision separately from the technical PostgreSQL `DATE` value.


In [8]:
DATE_FIELDS = [
    "startDateStruct",
    "primaryCompletionDateStruct",
    "completionDateStruct",
]

def date_precision(date_string):
    if not date_string:
        return None
    if re.fullmatch(r"\d{4}-\d{2}-\d{2}", date_string):
        return "DAY"
    if re.fullmatch(r"\d{4}-\d{2}", date_string):
        return "MONTH"
    return "OTHER"

for field in DATE_FIELDS:
    values = [
        get_nested(protocol, ("statusModule", field))
        for protocol in protocols
    ]
    values = [v for v in values if v is not None]

    types = Counter(v.get("type") for v in values if v.get("type") is not None)
    missing_type = sum(v.get("type") is None for v in values)
    precisions = Counter(
        date_precision(v.get("date"))
        for v in values
        if v.get("date") is not None
    )

    print(f"\n{field}")
    print("  present:", len(values))
    print("  types:", dict(types))
    print("  without type:", missing_type)
    print("  precision:", dict(precisions))



startDateStruct
  present: 199
  types: {'ACTUAL': 108, 'ESTIMATED': 24}
  without type: 67
  precision: {'DAY': 122, 'MONTH': 77}

primaryCompletionDateStruct
  present: 193
  types: {'ESTIMATED': 79, 'ACTUAL': 114}
  without type: 0
  precision: {'MONTH': 81, 'DAY': 112}

completionDateStruct
  present: 199
  types: {'ESTIMATED': 85, 'ACTUAL': 111}
  without type: 3
  precision: {'MONTH': 87, 'DAY': 112}


## 9. Enrollment validation


In [9]:
enrollment_values = [
    get_nested(protocol, ("designModule", "enrollmentInfo"))
    for protocol in protocols
]
enrollment_values = [v for v in enrollment_values if v is not None]

print("Enrollment records:", len(enrollment_values))
print("Types:", dict(Counter(
    v.get("type") for v in enrollment_values if v.get("type") is not None
)))
print("Without count:", sum(v.get("count") is None for v in enrollment_values))
print("Without type:", sum(v.get("type") is None for v in enrollment_values))


Enrollment records: 200
Types: {'ESTIMATED': 80, 'ACTUAL': 116}
Without count: 0
Without type: 4


## 10. Eligibility and age validation


In [10]:
AGE_PATTERN = re.compile(
    r"^\s*(?P<value>\d+(?:\.\d+)?)\s+(?P<unit>.+?)\s*$"
)

def parse_age(value):
    if value is None:
        return None, None
    match = AGE_PATTERN.match(value)
    if not match:
        return None, None
    return float(match.group("value")), match.group("unit")

for field in ["minimumAge", "maximumAge"]:
    parsed = [
        parse_age(get_nested(protocol, ("eligibilityModule", field)))
        for protocol in protocols
    ]
    parsed = [item for item in parsed if item[0] is not None]

    print(f"\n{field}")
    print("  present:", len(parsed))
    print("  units:", dict(Counter(unit for _, unit in parsed)))

print("\nStandardized ages:")
for value, count in collect_values(
    ("eligibilityModule", "stdAges"), True
).most_common():
    print(f"  {value}: {count}")

print("\nSex:")
for value, count in collect_values(
    ("eligibilityModule", "sex")
).most_common():
    print(f"  {value}: {count}")

print("\nHealthy volunteers:")
for value, count in collect_values(
    ("eligibilityModule", "healthyVolunteers")
).most_common():
    print(f"  {value}: {count}")

print(
    "Missing healthyVolunteers:",
    sum(
        get_nested(protocol, ("eligibilityModule", "healthyVolunteers")) is None
        for protocol in protocols
    )
)



minimumAge
  present: 189
  units: {'Years': 181, 'Hour': 1, 'Year': 2, 'Days': 1, 'Months': 2, 'Weeks': 1, 'Day': 1}

maximumAge
  present: 110
  units: {'Years': 106, 'Hours': 1, 'Days': 1, 'Weeks': 1, 'Months': 1}

Standardized ages:
  ADULT: 184
  OLDER_ADULT: 149
  CHILD: 45

Sex:
  ALL: 178
  FEMALE: 14
  MALE: 8

Healthy volunteers:
  False: 142
  True: 56
Missing healthyVolunteers: 2


## 11. Relationship cardinalities


In [11]:
RELATIONSHIP_PATHS = {
    "conditions": ("conditionsModule", "conditions"),
    "phases": ("designModule", "phases"),
    "interventions": ("armsInterventionsModule", "interventions"),
    "armGroups": ("armsInterventionsModule", "armGroups"),
    "locations": ("contactsLocationsModule", "locations"),
    "primaryOutcomes": ("outcomesModule", "primaryOutcomes"),
    "secondaryOutcomes": ("outcomesModule", "secondaryOutcomes"),
    "stdAges": ("eligibilityModule", "stdAges"),
}

for name, path in RELATIONSHIP_PATHS.items():
    counts = []
    for protocol in protocols:
        value = get_nested(protocol, path)
        counts.append(len(value) if isinstance(value, list) else 0)

    print(f"\n{name}")
    print("  trials with at least one:", sum(c > 0 for c in counts))
    print("  maximum per trial:", max(counts, default=0))
    print("  total records:", sum(counts))



conditions
  trials with at least one: 200
  maximum per trial: 12
  total records: 317

phases
  trials with at least one: 156
  maximum per trial: 2
  total records: 164

interventions
  trials with at least one: 182
  maximum per trial: 9
  total records: 333

armGroups
  trials with at least one: 179
  maximum per trial: 10
  total records: 380

locations
  trials with at least one: 179
  maximum per trial: 152
  total records: 1126

primaryOutcomes
  trials with at least one: 191
  maximum per trial: 21
  total records: 392

secondaryOutcomes
  trials with at least one: 141
  maximum per trial: 100
  total records: 789

stdAges
  trials with at least one: 200
  maximum per trial: 3
  total records: 378


## 12. Sponsor relationships


In [12]:
lead_count = 0
collaborator_counts = []

for protocol in protocols:
    sponsor_module = protocol.get("sponsorCollaboratorsModule", {})

    if sponsor_module.get("leadSponsor") is not None:
        lead_count += 1

    collaborator_counts.append(len(sponsor_module.get("collaborators", [])))

print("Trials with lead sponsor:", lead_count)
print("Trials with collaborators:", sum(c > 0 for c in collaborator_counts))
print("Total collaborators:", sum(collaborator_counts))
print("Maximum collaborators in one trial:", max(collaborator_counts, default=0))


Trials with lead sponsor: 200
Trials with collaborators: 56
Total collaborators: 99
Maximum collaborators in one trial: 6


## 13. Locations and outcomes


In [13]:
all_locations = []
all_outcomes = []

for protocol in protocols:
    contacts = protocol.get("contactsLocationsModule", {})
    outcome_module = protocol.get("outcomesModule", {})

    all_locations.extend(contacts.get("locations", []))

    for outcome in outcome_module.get("primaryOutcomes", []):
        all_outcomes.append({**outcome, "_outcome_type": "PRIMARY"})

    for outcome in outcome_module.get("secondaryOutcomes", []):
        all_outcomes.append({**outcome, "_outcome_type": "SECONDARY"})

print("Total locations:", len(all_locations))
for field in ["facility", "city", "state", "zip", "country", "geoPoint"]:
    present = sum(v.get(field) is not None for v in all_locations)
    print(f"{field:12} present={present:4} / {len(all_locations)}")

print("\nTotal outcomes:", len(all_outcomes))
for field in ["measure", "description", "timeFrame"]:
    present = sum(v.get(field) is not None for v in all_outcomes)
    print(f"{field:12} present={present:4} / {len(all_outcomes)}")

print("\nOutcome types:", dict(Counter(v["_outcome_type"] for v in all_outcomes)))


Total locations: 1126
facility     present= 711 / 1126
city         present=1126 / 1126
state        present= 718 / 1126
zip          present= 826 / 1126
country      present=1126 / 1126
geoPoint     present=1110 / 1126

Total outcomes: 1181
measure      present=1181 / 1181
description  present= 952 / 1181
timeFrame    present=1171 / 1181

Outcome types: {'PRIMARY': 392, 'SECONDARY': 789}


## 14. Arm groups and interventions

The current SQL uses `interventions`, `arm_groups`, and `arm_group_interventions`. This section validates the source multiplicities and the intervention labels attached to arm groups.


In [14]:
all_interventions = []
all_arm_groups = []

for protocol in protocols:
    module = protocol.get("armsInterventionsModule", {})
    all_interventions.extend(module.get("interventions", []))
    all_arm_groups.extend(module.get("armGroups", []))

print("Total interventions:", len(all_interventions))
print("Total arm groups:", len(all_arm_groups))

arms_with_labels = 0
label_count = 0

for arm in all_arm_groups:
    labels = arm.get("interventionNames")
    if labels:
        arms_with_labels += 1
        label_count += len(labels)

print("Arm groups with interventionNames:", arms_with_labels)
print("Total intervention labels in arm groups:", label_count)


Total interventions: 333
Total arm groups: 380
Arm groups with interventionNames: 348
Total intervention labels in arm groups: 456


## 15. Relevant fields outside the current relational scope

These fields are documented rather than silently forgotten. They are candidates for future extensions, but they are not required by the current SQL model.


In [15]:
OUT_OF_SCOPE_FIELDS = {
    "identificationModule": [
        "orgStudyIdInfo", "organization", "acronym", "secondaryIdInfos"
    ],
    "statusModule": [
        "statusVerifiedDate", "studyFirstSubmitDate", "studyFirstSubmitQcDate",
        "studyFirstPostDateStruct", "lastUpdateSubmitDate",
        "lastUpdatePostDateStruct", "expandedAccessInfo", "lastKnownStatus",
        "resultsFirstSubmitDate", "resultsFirstSubmitQcDate",
        "resultsFirstPostDateStruct", "whyStopped",
        "dispFirstSubmitDate", "dispFirstSubmitQcDate", "dispFirstPostDateStruct"
    ],
    "designModule": [
        "designInfo", "patientRegistry", "bioSpec", "targetDuration"
    ],
    "conditionsModule": ["keywords"],
    "sponsorCollaboratorsModule": ["responsibleParty"],
    "contactsLocationsModule": ["overallOfficials", "centralContacts"],
    "eligibilityModule": [
        "studyPopulation", "samplingMethod", "genderBased", "genderDescription"
    ],
    "outcomesModule": ["otherOutcomes"],
}

for module, fields in OUT_OF_SCOPE_FIELDS.items():
    print(f"\n{module}")
    for field in fields:
        print("  -", field)



identificationModule
  - orgStudyIdInfo
  - organization
  - acronym
  - secondaryIdInfos

statusModule
  - statusVerifiedDate
  - studyFirstSubmitDate
  - studyFirstSubmitQcDate
  - studyFirstPostDateStruct
  - lastUpdateSubmitDate
  - lastUpdatePostDateStruct
  - expandedAccessInfo
  - lastKnownStatus
  - resultsFirstSubmitDate
  - resultsFirstSubmitQcDate
  - resultsFirstPostDateStruct
  - whyStopped
  - dispFirstSubmitDate
  - dispFirstSubmitQcDate
  - dispFirstPostDateStruct

designModule
  - designInfo
  - patientRegistry
  - bioSpec
  - targetDuration

conditionsModule
  - keywords

sponsorCollaboratorsModule
  - responsibleParty

contactsLocationsModule
  - overallOfficials
  - centralContacts

eligibilityModule
  - studyPopulation
  - samplingMethod
  - genderBased
  - genderDescription

outcomesModule
  - otherOutcomes


## 16. Validation conclusions

After running the notebook, use the results to confirm:

1. optional modules are handled safely;
2. missing scalar fields become `NULL`;
3. missing child lists produce no child records;
4. `NA` phases are ignored;
5. partial dates preserve their precision;
6. age values are separated into numeric value and unit;
7. standardized ages use a many-to-many relationship;
8. locations and outcomes use one-to-many relationships;
9. sponsor roles are represented through `trial_sponsors.role`;
10. fields outside the current analytical scope are explicitly documented.

If a new value or structure is incompatible with the current schema, update the Data Model Notes before implementing ingestion.

## 17. Next step

Once validation is stable, the project can move to the ingestion layer:

- retrieve records with pagination;
- transform source structures into relational rows;
- insert lookup values;
- insert trials;
- insert bridge-table relationships;
- insert child records;
- validate foreign keys and row counts;
- run database-level quality checks.

This notebook should remain as a regression check when the ingestion pipeline changes.
